In [1]:
import requests
import datetime
import pandas as pd


In [2]:


import time

API_KEY = "aa6877ac64bbbc776b89c98b61b11b54"
lat = 24.8607
lon = 67.0011

# Define start and end dates
end_date = pd.to_datetime("today").normalize()
start_date = end_date - pd.DateOffset(years=1)

# Convert to timestamps
start_ts = int(start_date.timestamp())
end_ts = int(end_date.timestamp())

# OpenWeatherMap allows limited range per request, so chunk by 30 days
chunk_days = 30
chunk_seconds = chunk_days * 24 * 60 * 60

all_data = []

current_start = start_ts
while current_start < end_ts:
    current_end = min(current_start + chunk_seconds, end_ts)
    
    params = {
        'lat': lat,
        'lon': lon,
        'start': current_start,
        'end': current_end,
        'appid': API_KEY
    }
    
    response = requests.get("http://api.openweathermap.org/data/2.5/air_pollution/history", params=params)
    response.raise_for_status()
    data = response.json()
    
    if 'list' in data:
        all_data.extend(data['list'])
    
    print(f"Fetched data from {current_start} to {current_end}, total records: {len(data.get('list', []))}")
    current_start = current_end + 1
    time.sleep(1)  # avoid hitting rate limits

# Convert to DataFrame



Fetched data from 1739318400 to 1741910400, total records: 673
Fetched data from 1741910401 to 1744502401, total records: 552
Fetched data from 1744502402 to 1747094402, total records: 696
Fetched data from 1747094403 to 1749686403, total records: 720
Fetched data from 1749686404 to 1752278404, total records: 720
Fetched data from 1752278405 to 1754870405, total records: 720
Fetched data from 1754870406 to 1757462406, total records: 720
Fetched data from 1757462407 to 1760054407, total records: 720
Fetched data from 1760054408 to 1762646408, total records: 720
Fetched data from 1762646409 to 1765238409, total records: 720
Fetched data from 1765238410 to 1767830410, total records: 696
Fetched data from 1767830411 to 1770422411, total records: 696
Fetched data from 1770422412 to 1770854400, total records: 120


In [3]:
raw_df = pd.DataFrame(all_data)
raw_df['datetime'] = pd.to_datetime(raw_df['dt'], unit='s')
raw_df.head()

,main,components,dt,datetime
0,{'aqi': 5},"{'co': 2376.56, 'no': 0.07, 'no2': 69.92, 'o3'...",1739318400,2025-02-12 00:00:00
1,{'aqi': 5},"{'co': 2563.48, 'no': 0.15, 'no2': 72.66, 'o3'...",1739322000,2025-02-12 01:00:00
2,{'aqi': 5},"{'co': 2830.51, 'no': 1.22, 'no2': 80.2, 'o3':...",1739325600,2025-02-12 02:00:00
3,{'aqi': 5},"{'co': 4058.84, 'no': 24.36, 'no2': 100.08, 'o...",1739329200,2025-02-12 03:00:00
4,{'aqi': 5},"{'co': 6034.85, 'no': 82.25, 'no2': 115.16, 'o...",1739332800,2025-02-12 04:00:00


In [4]:
df_main = raw_df['main'].apply(pd.Series)
df_components = raw_df['components'].apply(pd.Series)
df_components.head()

,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2376.56,0.07,69.92,16.45,19.31,141.36,219.46,11.78
1,2563.48,0.15,72.66,14.66,21.46,153.87,236.32,12.92
2,2830.51,1.22,80.20,8.31,23.84,161.20,246.97,17.73
3,4058.84,24.36,100.08,0.87,30.28,210.00,301.09,32.42
4,6034.85,82.25,115.16,5.99,39.58,279.51,380.58,54.21


In [5]:
df = pd.concat([raw_df.drop(['main', 'components'], axis=1), df_main, df_components], axis=1)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1739318400,2025-02-12 00:00:00,5.0,2376.56,0.07,69.92,16.45,19.31,141.36,219.46,11.78
1,1739322000,2025-02-12 01:00:00,5.0,2563.48,0.15,72.66,14.66,21.46,153.87,236.32,12.92
2,1739325600,2025-02-12 02:00:00,5.0,2830.51,1.22,80.20,8.31,23.84,161.20,246.97,17.73
3,1739329200,2025-02-12 03:00:00,5.0,4058.84,24.36,100.08,0.87,30.28,210.00,301.09,32.42
4,1739332800,2025-02-12 04:00:00,5.0,6034.85,82.25,115.16,5.99,39.58,279.51,380.58,54.21


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8473 entries, 0 to 8472
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype        
---  ------    --------------  -----        
 0   dt        8473 non-null   int64        
 1   datetime  8473 non-null   datetime64[s]
 2   aqi       8473 non-null   float64      
 3   co        8473 non-null   float64      
 4   no        8473 non-null   float64      
 5   no2       8473 non-null   float64      
 6   o3        8473 non-null   float64      
 7   so2       8473 non-null   float64      
 8   pm2_5     8473 non-null   float64      
 9   pm10      8473 non-null   float64      
 10  nh3       8473 non-null   float64      
dtypes: datetime64[s](1), float64(9), int64(1)
memory usage: 728.3 KB


In [7]:
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,dt,datetime,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1739318400,2025-02-12 00:00:00,5.0,2376.56,0.07,69.92,16.45,19.31,141.36,219.46,11.78
1,1739322000,2025-02-12 01:00:00,5.0,2563.48,0.15,72.66,14.66,21.46,153.87,236.32,12.92
2,1739325600,2025-02-12 02:00:00,5.0,2830.51,1.22,80.20,8.31,23.84,161.20,246.97,17.73
3,1739329200,2025-02-12 03:00:00,5.0,4058.84,24.36,100.08,0.87,30.28,210.00,301.09,32.42
4,1739332800,2025-02-12 04:00:00,5.0,6034.85,82.25,115.16,5.99,39.58,279.51,380.58,54.21


In [8]:
df.to_csv("../Dataset/aqi_data.csv", index=False)

In [9]:
df['aqi'].value_counts()


aqi
3.000000    3489
4.000000    2242
2.000000    1222
5.000000    1052
1.000000     466
3.333332       1
5.555580       1
Name: count, dtype: int64